# Intelligent Loan Approval - EDA & Model Training
This notebook handles the data preprocessing, model training (Random Forest), evaluation, and exporting of the model artifacts to your Google Drive.

In [1]:
!pip install xgboost imbalanced-learn shap -q

In [13]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report, roc_auc_score, confusion_matrix
from imblearn.over_sampling import SMOTE
import shap
from google.colab import drive

## 1. Mount Google Drive

In [3]:
drive.mount('/content/drive')
# Create the destination directory if it doesn't exist
drive_path = '/content/drive/MyDrive/Intelligent Loan Approval'
os.makedirs(drive_path, exist_ok=True)

Mounted at /content/drive


## 2. Load Data
**IMPORTANT**: Please ensure you have uploaded the `loan_dataset.csv` (Credit Risk Dataset from Kaggle) into your Colab environment or Google Drive and update the path below if necessary.

In [6]:
# Assuming the dataset is uploaded to the root of the Colab session
df = pd.read_csv('/content/loan_dataset.csv')
df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


## 3. Data Preprocessing

In [8]:
# Drop nulls (or impute them)
df = df.dropna()
df = df.drop_duplicates()

# Separate features and target (Assuming target is 'loan_status')
X = df.drop('loan_status', axis=1)
y = df['loan_status']

numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

# 1. Scale numerical features
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X[numerical_cols])

# 2. Encode categorical features
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
X_cat_encoded = encoder.fit_transform(X[categorical_cols])
cat_feature_names = encoder.get_feature_names_out(categorical_cols)

# Combine features
X_processed = np.hstack((X_num_scaled, X_cat_encoded))
feature_columns = numerical_cols + list(cat_feature_names)

print(f'Processed shape: {X_processed.shape}')

Processed shape: (28501, 26)


## 4. Train/Test Split and Handling Class Imbalance

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42, stratify=y)

print("Before SMOTE:")
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE:")
print(y_train_resampled.value_counts())

Before SMOTE:
loan_status
0    17850
1     4950
Name: count, dtype: int64

After SMOTE:
loan_status
0    17850
1    17850
Name: count, dtype: int64


## 5. Model Training (Random Forest)

In [10]:
model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=15)
model.fit(X_train_resampled, y_train_resampled)

RandomForestClassifier(max_depth=15, random_state=42)

## 6. Evaluation

In [11]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.98      0.95      4463
           1       0.92      0.72      0.81      1238

    accuracy                           0.92      5701
   macro avg       0.92      0.85      0.88      5701
weighted avg       0.92      0.92      0.92      5701

ROC-AUC Score: 0.9293

Confusion Matrix:
[[4384   79]
 [ 349  889]]


## 7. Model Export to Google Drive

In [14]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}")

if accuracy >= 0.90:
    user_choice = input(f"Model accuracy is {accuracy:.2%} (>= 90%). Do you want to export the models to Google Drive? (yes/no): ")
    if user_choice.strip().lower() in ['yes', 'y']:
        joblib.dump(model, f"{drive_path}/loan_model.pkl")
        joblib.dump(scaler, f"{drive_path}/scaler.pkl")
        joblib.dump(encoder, f"{drive_path}/encoder.pkl")
        joblib.dump(feature_columns, f"{drive_path}/feature_columns.pkl")
        print(f"Models successfully exported to: {drive_path}")
    else:
        print("Export cancelled by user.")
else:
    print(f"Model accuracy is {accuracy:.2%} which is below the 90% threshold. Models will not be exported.")

Model Accuracy: 0.9249
Model accuracy is 92.49% (>= 90%). Do you want to export the models to Google Drive? (yes/no): yes
Models successfully exported to: /content/drive/MyDrive/Intelligent Loan Approval
